# VieNeu-TTS v3 Turbo trên Kaggle T4 ×2

Bật **Internet** và **GPU T4 ×2** trước khi chạy. Chạy các cell theo thứ tự. Giữ notebook đang chạy để Hugging Face Space gọi API. Mỗi GPU phục vụ một worker riêng; model chạy FP32 vì FP16 trên T4 chưa được tác giả kiểm chứng chất lượng.


In [ ]:
!pip -q install 'vieneu[cuda]==3.8.1' 'transformers==4.57.6' 'fastapi>=0.115,<1' 'uvicorn>=0.30,<1' 'httpx>=0.27,<1' 'python-multipart>=0.0.9' 
!pip -q uninstall -y torchvision


In [ ]:
import torch
print('CUDA GPUs:', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() == 2, 'Select Kaggle GPU T4 x2 before running this notebook'


In [ ]:
import pathlib, urllib.request
source = 'https://huggingface.co/spaces/phucsd/vieneu-tts-kaggle/resolve/main/kaggle_backend.py'
destination = pathlib.Path('/kaggle/working/kaggle_backend.py')
urllib.request.urlretrieve(source, destination)
print('Backend saved to', destination)


In [ ]:
import os, secrets, subprocess, sys, time, requests, pathlib

api_key = secrets.token_urlsafe(32)
base_env = os.environ.copy()
base_env['VIENEU_API_KEY'] = api_key
processes = []
logs = []

for gpu_id, port in [(0, 8001), (1, 8002)]:
    env = base_env.copy()
    env['CUDA_VISIBLE_DEVICES'] = str(gpu_id)
    env['VIENEU_ROLE'] = 'worker'
    log = open(f'/kaggle/working/vieneu_gpu{gpu_id}.log', 'w')
    process = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'kaggle_backend:app',
                                '--host', '127.0.0.1', '--port', str(port)],
                               cwd='/kaggle/working', env=env, stdout=log, stderr=subprocess.STDOUT)
    processes.append(process)
    logs.append(log)

gateway_env = base_env.copy()
gateway_env['VIENEU_ROLE'] = 'gateway'
gateway_log = open('/kaggle/working/vieneu_gateway.log', 'w')
gateway = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'kaggle_backend:app',
                            '--host', '127.0.0.1', '--port', '8000'],
                           cwd='/kaggle/working', env=gateway_env,
                           stdout=gateway_log, stderr=subprocess.STDOUT)
processes.append(gateway)
headers = {'Authorization': 'Bearer ' + api_key}

deadline = time.monotonic() + 360
while time.monotonic() < deadline:
    if any(process.poll() is not None for process in processes):
        raise RuntimeError('Server failed. Read /kaggle/working/vieneu_gpu*.log and vieneu_gateway.log')
    try:
        health = requests.get('http://127.0.0.1:8000/health', headers=headers, timeout=6).json()
        print('GPU workers ready:', health['ready_workers'], '/', health['total_workers'])
        if health['ready_workers'] == 2:
            break
    except (requests.RequestException, ValueError, KeyError):
        pass
    time.sleep(5)
else:
    raise TimeoutError('Workers did not load within 6 minutes. Check the log files.')


In [ ]:
import re, urllib.request

binary = pathlib.Path('/kaggle/working/cloudflared')
if not binary.exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', binary)
    binary.chmod(0o755)

tunnel_log = open('/kaggle/working/vieneu_tunnel.log', 'w')
tunnel = subprocess.Popen([str(binary), 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8000'],
                          stdout=tunnel_log, stderr=subprocess.STDOUT)
deadline = time.monotonic() + 60
api_url = None
while time.monotonic() < deadline:
    tunnel_log.flush()
    contents = pathlib.Path('/kaggle/working/vieneu_tunnel.log').read_text(errors='replace')
    found = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', contents)
    if found:
        api_url = found.group(0)
        break
    if tunnel.poll() is not None:
        raise RuntimeError('Tunnel stopped. Read /kaggle/working/vieneu_tunnel.log')
    time.sleep(2)
if not api_url:
    raise TimeoutError('Tunnel URL not found. Read /kaggle/working/vieneu_tunnel.log')

print('API URL:', api_url)
print('API key:', api_key)
print('Space: https://huggingface.co/spaces/phucsd/vieneu-tts-kaggle')


In [ ]:
# Verify inference through the local gateway; the public tunnel is checked separately.
test = requests.post('http://127.0.0.1:8000/synthesize', headers=headers,
                     data={'text': 'Xin chào, đây là bản thử nghiệm VieNeu.', 'voice': 'Minh Quân Pro'},
                     timeout=300)
print('HTTP status:', test.status_code, 'Content type:', test.headers.get('content-type'))
test.raise_for_status()
pathlib.Path('/kaggle/working/vieneu_test.wav').write_bytes(test.content)
from IPython.display import Audio, display
display(Audio('/kaggle/working/vieneu_test.wav'))


In [ ]:
# Keep the Kaggle kernel and its two GPU workers running until you stop this cell or Kaggle ends the session.
print('Serving from', api_url)
while tunnel.poll() is None and all(process.poll() is None for process in processes):
    time.sleep(20)
raise RuntimeError('A worker or tunnel stopped. Check the log files in /kaggle/working.')
